# Fundamentals 13 - Multi AgenticSystem Completo

Sistema completo multi-agente: solver, judge y reviewer LM opcional. Incluye pipeline, graph local, environment, eval y lineage.


In [ ]:
import os
import agentic_systems as lab
PRETTY = False
scheduler = lab.scheduler(timeout_s=60, max_retries=0, max_tool_calls=6, max_turns=6)
local_runtime = lab.runtime(provider="python-direct", model="local-python", region="local", scheduler=scheduler)
lm_runtime = lab.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
workspace = lab.AgenticSystem(model=lm_runtime.model_id or "local-python", region=lm_runtime.region_name or "local", runtime=lm_runtime)
USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42
lab.show({"runtime_auto": lm_resolution, "lm_available": lm_available, "force_local_only": force_local_only})


In [ ]:
@lab.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b; x2 = x1 - c; x3 = x2 * d; x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}
@lab.tool
def judge_result(result: float, expected: float) -> dict:
    ok = float(result) == float(expected)
    return {"ok": ok, "score": 1.0 if ok else 0.0, "result": result, "expected": expected}


## 1) Agentes del sistema


In [ ]:
@lab.tool
def record_review(summary: str) -> dict:
    """Registra una revisi?n LM como evidencia estructurada."""
    return {"summary": summary}

policy = lab.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
solver = lab.agent(name="multi_solver", instructions="Resuelve n?meros estructurados.", tools=[solve_arithmetic], engine="python-direct", runtime=local_runtime, contract=lab.AgentContract(must_call=["solve_arithmetic"], tool_expectation=lab.expect.exactly("solve_arithmetic")), policy=policy)
judge = lab.agent(name="multi_judge", instructions="Valida resultado.", tools=[judge_result], engine="python-direct", runtime=local_runtime, contract=lab.AgentContract(must_call=["judge_result"], tool_expectation=lab.expect.exactly("judge_result")), policy=policy)
reviewer = workspace.agent(name="multi_lm_reviewer", instructions="Revisa evidencia final sin cambiar n?meros.", tools=[record_review], runtime=lm_runtime, policy=lab.RunPolicy.for_mode("eval"))
lab.show({"agents": [solver.info(), judge.info(), reviewer.info()]})


## 2) Pipeline multi-agente + lineage


In [ ]:
solve = solver.run({"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}})
judgement = judge.run({"tool": "judge_result", "input": {"result": solve.data["result"], "expected": EXPECTED}})
review = None
if lm_available:
    review = reviewer.run(str({"solution": solve.data, "judge": judgement.data}))
else:
    lab.show({"status": "skipped", "reason": lm_resolution["reason"]}, title="LM reviewer saltado")

final = {
    "procedimiento": solve.data["procedure"],
    "resultado_final": solve.data["result"],
    "judge": judgement.data,
    "lm_review": review.text if review else None,
}
result = lab.compose_result(
    text="Multi AgenticSystem completo ejecutado.",
    data=final,
    results=[solve, judgement, review],
    mode="multi-agentic-system",
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution},
)
lineage = result.lineage(name="fundamentals.multi_agentic_system_full", question=USER_PROMPT, goal="Explicar pipeline multi-agente completo.")
lab.human_result(result, title="Human result - Multi AgenticSystem", pretty=PRETTY, show_lineage=True, lineage=lineage)

## 3) Graph local + environment + eval


In [ ]:
def node_solve(state: dict) -> dict:
    out = solver.run({"tool": "solve_arithmetic", "input": {"numbers": state["numbers"]}}).data
    return {**state, "procedure": out["procedure"], "result": out["result"]}
def node_judge(state: dict) -> dict:
    out = judge.run({"tool": "judge_result", "input": {"result": state["result"], "expected": EXPECTED}}).data
    return {**state, "judge": out}
graph_state = {"numbers": NUMBERS}
for node in [node_solve, node_judge]:
    graph_state = node(graph_state)
def transition_fn(row: dict, action: dict | None, info: dict) -> dict:
    solved = solver.run({"tool": "solve_arithmetic", "input": {"numbers": row["numbers"]}}).data
    judged = judge.run({"tool": "judge_result", "input": {"result": solved["result"], "expected": row["expected"]}}).data
    return {"result": solved["result"], "judge": judged}
def reward_fn(state: dict) -> float:
    return 1.0 if (state.get("judge") or {}).get("ok") else 0.0
env_records = [{"numbers": NUMBERS, "expected": EXPECTED}]
env = lab.AgenticEnvironment(name="multi_system_env", records=env_records, initial_memory={}, transition_fn=transition_fn, reward_fn=reward_fn)
env.reset()
_, reward, terminated, truncated, info = env.step()
report = lab.run_eval(solver, [{"id": "default", "input": {"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}}, "expected": {"result": EXPECTED}}])
lab.show({"graph_state": graph_state, "env_summary": env.summary(), "step": info["transition"], "eval_report": report.to_dict()})


In [ ]:
lab.show({"notebook": "13_multi agentic_system.ipynb", "api_coverage": ["multi-agent pipeline", "runtime(provider='auto')", "graph nodes", "AgenticEnvironment", "run_eval", "compose_result", "RunResult.lineage"]})
